<a href="https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026_10_16_Jun_Seo_Lee_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [139]:
import pandas as pd
PATH = 'notebooks/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [140]:
# TODO (hint: .str.lower().str.contains('pop'))
poptarts = cat[cat['product_name'].str.lower().str.contains('pop', na=False)].copy()

print(f'Number of Pop-Tart products: {len(poptarts)}')
poptarts

Number of Pop-Tart products: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [141]:
# TODO
strawberry_poptarts = poptarts[poptarts['product_name'].str.lower().str.contains('straw')]
non_strawberry_poptarts = poptarts[~poptarts['product_name'].str.lower().str.contains('straw')]

print(f"Strawberry Pop-Tart variants: {len(strawberry_poptarts)}")
display(strawberry_poptarts[['sku', 'product_name']])

print(f"Non-strawberry Pop-Tart variants: {len(non_strawberry_poptarts)}")
display(non_strawberry_poptarts[['sku', 'product_name']])

Strawberry Pop-Tart variants: 3


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry


Non-strawberry Pop-Tart variants: 0


,sku,product_name


**Pop-Tart rows that are a different product:** _..._

Strawberry Pop-Tart variants:  3

Non-Strawberry Pop-Tart variants:  0

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [142]:
# TODO
CANON_SKU = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',}

consolidated = cat.copy()

consolidated['canonical_sku'] = (consolidated['sku'].map(CANON_SKU).fillna(consolidated['sku'])
)

consolidated

,sku,product_name,category,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [143]:
# TODO
CANON_CAT = {
    'Breakfast Foods': 'Breakfast',
    'Breakfast': 'Breakfast',}

print("Categories before cleanup:")
print(consolidated['category'].value_counts(dropna=False).sort_index())

consolidated['category_clean'] = (consolidated['category'].str.strip().str.title())
print("\nCategories after strip and title-case:")
print(consolidated['category'].value_counts(dropna=False).sort_index())

consolidated['canonical_cat'] = (consolidated['category'].map(CANON_CAT).fillna(consolidated['category'])
)
print("\nCategories after cleanup:")
print(consolidated['canonical_cat'].value_counts(dropna=False).sort_index())

Categories before cleanup:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

Categories after strip and title-case:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

Categories after cleanup:
canonical_cat
Breakfast    3
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [144]:
from os.path import join
# TODO
consolidated_analysis = consolidated.copy()

consolidated_analysis['product_name_clean'] = (consolidated_analysis['product_name'].str.lower().str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip().apply(lambda name: ' '.join(sorted(name.split()))))

consolidated_analysis_groups = (consolidated_analysis.groupby('product_name_clean').agg(sku_count=('sku', 'nunique'), skus=('sku', list), product_names=('product_name', list))).reset_index()

consolidated_analysis_duplicated = consolidated_analysis_groups.loc[consolidated_analysis_groups['sku_count'] > 1].copy()

consolidated_analysis_duplicated.sort_values('sku_count', ascending=False)

,product_name_clean,sku_count,skus,product_names
1,pop strawberry tarts,2,"[POPTART-STRAW, POPTART-STRAW]","[pop-tarts strawberry, Pop-Tarts Strawberry]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [145]:
# TODO

assert consolidated['canonical_sku'].notna().all()
assert len(consolidated) == len(cat)

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.



**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two fragmented product families would be the AA batteries and Monopoly. They appear to have different SKU names to identify the same products. I would group these under singular SKU names such as BATT_AA and GAME_MONO for example. However, there are obviously things that oculd be different such as different versions of the game, different sizes/numbers of batteries, etc. that need to be checked before these products are grouped.

b) Consolidated becomes a business decision when the available data can no longer prove that two records represent the same product. Differences in product models, sizes, etc. require intimate domain and business knowledge. For example, with the battery example I gave earlier, there is a possibility that all the battery products are the same amount, package size, etc. however, it could be possible that they actually dome in different sizes as is presented in the package_size description column but not reflected within the product_name and SKU columns. There needs to be more clarification and confirmation done to ensure that these are actually the same products.